# 0. Definitions

## 0.0. REquired Packages

In [ ]:
# file: simulator/onequbit.py

import matplotlib.pyplot as plt
from matplotlib import font_manager
from pylab import *
import scipy
from scipy.optimize import minimize

from scipy.linalg import expm

from typing import Callable, Any
from matplotlib.collections import LineCollection
import numpy as np

import numpy as np
import time

import dataclasses

Vector = np.ndarray

## 0.1. WW Theory (Yang's calculation)

In [ ]:
def C_evolution(C: np.array, c0: np.array, times: np.array):
    c_t = [[],[],[]]
    for t in times:
        c_evolved = np.dot(expm(C*t),c0)
        c_t[0].append(c_evolved[0])
        c_t[1].append(c_evolved[1])
        c_t[2].append(c_evolved[2])
    return np.array(c_t)

In [ ]:
from scipy.integrate import solve_ivp


def evolve_C_time_dependent(c0, times, gamma, phi, J, omega_d, β, Δ):

    def ode(t, c_flat):
        # c_flat is a 1D complex vector (size 3)
        C_t = C_of_t(t, gamma, phi, J, omega_d, β, Δ)
        return C_t @ c_flat

    sol = solve_ivp(
        ode,
        (times[0], times[-1]),
        c0.astype(complex),
        t_eval=times,
        method='RK45'
    )

    return sol.y

def C_of_t(t, gamma, phi, J, omega_d, β, Δ):
    Δ_t = Δ*np.cos(omega_d * t + β)  # Δ(t)

    # Build full C(t)
    C = np.array([
        [-gamma/2,                    -gamma*np.exp(1j*phi)/2,     -1j*J],
        [-gamma*np.exp(1j*phi)/2,     -gamma/2,                    -1j*J],
        [-1j*J,                       -1j*J,                       -1j*Δ_t]
    ], dtype=complex)

    return C

def LogicalBit_pop(Cs: np.array, times: np.array):
    P0_L, P1_L = [], []
    for t in range(len(times)):
        P0_L.append(abs( (1/2)*(Cs[0][t] + Cs[1][t]) - Cs[2][t]/np.sqrt(2) )**2)
        P1_L.append(abs( (1/2)*(Cs[0][t] + Cs[1][t]) + Cs[2][t]/np.sqrt(2) )**2)
    return [np.array(P0_L), np.array(P1_L)]

def LogicalPlusMinus_pop(Cs: np.array, times: np.array):
    Plus_L, Minus_L = [], []
    for t in range(len(times)):
        Plus_L.append(abs( Cs[2][t] )**2)
        Minus_L.append((1/2)*abs( (Cs[0][t] + Cs[1][t])  )**2)
    return [np.array(Plus_L), np.array(Minus_L)]

def LogicalPhase(Cs: np.array, times: np.array, RotatingFrame = 'False'):
    sinf_I_0, sinf_R_0 = [], []
    sinf_I_1, sinf_R_1 = [], []
    if RotatingFrame == 'True':
        for t in range(len(times)):
            coe_I_0 = np.imag(np.exp(1.j*4*times[t])*((1/2)*(Cs[0][t] + Cs[1][t]) + Cs[2][t]/np.sqrt(2)))
            coe_R_0 = np.real(np.exp(1.j*4*times[t])*((1/2)*(Cs[0][t] + Cs[1][t]) + Cs[2][t]/np.sqrt(2)))
            sinf_I_0.append(coe_I_0*np.sqrt(2))
            sinf_R_0.append(coe_R_0*np.sqrt(2))
            coe_I_1 = np.imag(np.exp(1.j*4*times[t])*((1/2)*(Cs[0][t] + Cs[1][t]) - Cs[2][t]/np.sqrt(2)))
            coe_R_1 = np.real(np.exp(1.j*4*times[t])*((1/2)*(Cs[0][t] + Cs[1][t]) - Cs[2][t]/np.sqrt(2)))
            sinf_I_1.append(coe_I_1*np.sqrt(2))
            sinf_R_1.append(coe_R_1*np.sqrt(2))
    else:
        for t in range(len(times)):
            coe_I_0 = np.imag(((1/2)*(Cs[0][t] + Cs[1][t]) + Cs[2][t]/np.sqrt(2)))
            coe_R_0 = np.real(((1/2)*(Cs[0][t] + Cs[1][t]) + Cs[2][t]/np.sqrt(2)))
            sinf_I_0.append(coe_I_0*np.sqrt(2))
            sinf_R_0.append(coe_R_0*np.sqrt(2))
            coe_I_1 = np.imag(((1/2)*(Cs[0][t] + Cs[1][t]) - Cs[2][t]/np.sqrt(2)))
            coe_R_1 = np.real(((1/2)*(Cs[0][t] + Cs[1][t]) - Cs[2][t]/np.sqrt(2)))
            sinf_I_1.append(coe_I_1*np.sqrt(2))
            sinf_R_1.append(coe_R_1*np.sqrt(2))
    return [[sinf_I_0,sinf_R_0],[sinf_I_1,sinf_R_1]]

def compute_eta(phi, Delta, gamma0, J, times):
    # Matrix C
    C = np.array([
        [-gamma0/2, -gamma0*np.exp(1j*phi)/2, -1j*J],
        [-gamma0*np.exp(1j*phi)/2, -gamma0/2, -1j*J],
        [-1j*J, -1j*J, -1j*Delta]
    ])

    # Initial state
    c0 = np.array([
        0.5,
        0.5*np.exp(1j*(np.pi - phi)),
        -np.sqrt(0.5)
    ])

    # Time evolution (your existing function)
    C1, C2, C3 = C_evolution(C, c0, times)

    # Left and right emission
    alpha_L = np.abs(np.sqrt(gamma0/2) * (C1 + np.exp(1j*phi)*C2))**2
    alpha_R = np.abs(np.sqrt(gamma0/2) * (C1 + np.exp(-1j*phi)*C2))**2

    # Time integrals
    I_l = np.trapezoid(alpha_L, times)
    I_r = np.trapezoid(alpha_R, times)

    # Directionality
    eta = (I_r - I_l) / (I_r + I_l)

    return [eta.real, I_r + I_l]

def objective(x, gamma0, J, times):
    phi, Delta = x
    return -compute_eta(phi, Delta, gamma0, J, times)[0]

def compute_eta_Logical1(phi, Delta, gamma0, J, times):
    # Matrix C
    C = np.array([
        [-gamma0/2, -gamma0*np.exp(1j*phi)/2, -1j*J],
        [-gamma0*np.exp(1j*phi)/2, -gamma0/2, -1j*J],
        [-1j*J, -1j*J, -1j*Delta]
    ])

    # Initial state
    c0 = np.array([
        0.5,
        0.5*np.exp(1j*(np.pi - phi)),
        -np.sqrt(0.5)
    ])

    # Time evolution (your existing function)
    C1, C2, C3 = C_evolution(C, c0, times)

    # Left and right emission
    alpha_L = np.abs(np.sqrt(gamma0/2) * (C1 + np.exp(1j*phi)*C2))**2
    alpha_R = np.abs(np.sqrt(gamma0/2) * (C1 + np.exp(-1j*phi)*C2))**2

    # Time integrals
    I_l = np.trapezoid(alpha_L, times)
    I_r = np.trapezoid(alpha_R, times)

    # Directionality
    eta = (I_r - I_l) / (I_r + I_l)

    return [eta.real, I_r + I_l]

def objective_Logical1(x, gamma0, J, times):
    phi, Delta = x
    return -compute_eta_Logical1(phi, Delta, gamma0, J, times)[0]

# 1. Decoherence Free Qubit Operations

## 1.1. Quantum Gates

### Hadamard gate

In [ ]:
# Parameters Definition
#GHZ = 1

ϕ = np.pi
γ = 0.1
ω0 = 4.0
phase = np.asarray([0, ϕ, 0])
positions = phase / ω0
Nqubits = 3

# State1 = 010
# State1 = 001

J = 0.5*γ
Δ = 2*np.sqrt(2)*J #Hadamard setup
ε = np.pi
θ = 0

tH = np.pi/(4*J)
times = np.linspace(0,2*tH,200)

In [ ]:
#WW theory calculations

C = np.array([[-γ/2, -γ*np.exp(1.j*ϕ)/2 , - 1.j*J ],[-γ*np.exp(1.j*ϕ)/2,-γ/2, - 1.j*J ],[- 1.j*J,- 1.j*J, -1.j*Δ ]])

cL0 = np.array([1/2, 1/2,- np.sqrt(0.5)])
cL1 = np.array([1/2, 1/2,+ np.sqrt(0.5)])
c0 = np.array([np.cos(ε/2)*np.sqrt(0.5), np.cos(ε/2)*np.sqrt(0.5)*np.exp(1.j*θ), - np.sin(ε/2)])
C1, C2, C3 = C_evolution(C,cL0,times)

C_WW = [C1, C2, C3]

Pops_WW = [abs(C1)**2, abs(C2)**2, abs(C3)**2]

In [ ]:

LogicBasis_WW = LogicalBit_pop(C_WW, times)

PMBasis_WW = LogicalPlusMinus_pop(C_WW, times)


In [ ]:

figsize = (9, 6)
fig, axs = plt.subplots(2, 2, figsize=figsize)
fig.subplots_adjust(hspace=0.5, wspace=0.4)
csfont = {"fontname": "Times New Roman"}
Legendfont = font_manager.FontProperties(math_fontfamily="cm", size=10)
Labels_FontSize = 18
Box_width = 2

Colors= [
        ["#08589e", "#4eb3d3"],
        ["#91003f", "#c994c7"],
        ["#006600", "#99FF99"],
        ["#cc4c02", "#fe9929"],
        ["black", "#E0E0E0"],
        ["#990000", "#FFCCCC"]
        ]
Marks = ["o", "s", "D", "p", "8", "v", "p", "v", "^", "8"]
M_size = [6,6,6,6]




ind_g = [(0,0),(0,1),(1,0)]


for ind in range(0,2):
    l1, l2 = 0,0
    axs[l1][l2].set_ylabel(
        r"$P_{|n_{L}\rangle}$", fontsize=Labels_FontSize, math_fontfamily="cm", **csfont
    )
    #axs[l1][l2].plot(times/tH, PMBasis_exact[ind], label=r"$\mathrm{ED}$", linewidth=3, ls="-",color=Colors[ind][0])
    axs[l1][l2].plot(times/tH, PMBasis_WW[ind], label=r"$\mathrm{WW}$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=10)


for l1 in [0, 1]:
    for l2 in [0, 1]:
        for label in axs[l1][l2].get_xticklabels() + axs[l1][l2].get_yticklabels():
            label.set_fontname("Times New Roman")
        axs[l1][l2].legend(prop=Legendfont, loc="best")
        axs[l1][l2].set_xlabel(r"$t / \tau_{H}$", fontsize=Labels_FontSize, math_fontfamily="cm", **csfont)
        axs[l1][l2].xaxis.set_tick_params(width=Box_width, length=6, labelsize=Labels_FontSize)
        axs[l1][l2].yaxis.set_tick_params(width=Box_width, length=6, labelsize=Labels_FontSize)
        axs[l1][l2].tick_params(axis="y", which="minor", width=Box_width, length=4)
        axs[l1][l2].tick_params(axis="x", which="minor", width=Box_width, length=4)
        axs[l1][l2].set_ylim(-0.03, 1.03)
        axs[l1][l2].set_xlim(-0.02, 2.03)
        axs[l1][l2].yaxis.set_major_locator(MultipleLocator(0.2))
        axs[l1][l2].xaxis.set_major_locator(MultipleLocator(0.5))
        axs[l1][l2].xaxis.set_major_formatter("{x:.1f}")
        axs[l1][l2].yaxis.set_major_formatter("{x:.1f}")
        plt.setp(axs[l1][l2].spines.values(), linewidth = Box_width)

plt.savefig("HadamardGate.pdf")

# 2. Chiral Quantum Optics.

## 2.1. Chiral photon emission

In [ ]:
x0 = [0.0, 0.0]  # initial guess: phi, Delta
γ0 = 0.1
J = 0.1*γ0
times = np.linspace(0,500,30000)/γ0

result = minimize(
    objective,
    x0,
    args=(γ0, J, times),
    method='Nelder-Mead'
)

φ_opt, Δ_opt= result.x
eta_max = -result.fun

φ_opt = φ_opt/np.pi
Δ_opt = Δ_opt/γ0
print(eta_max)

print('Optimal φ =', φ_opt)
print('Optimal Δ =', Δ_opt)

In [ ]:
#Defining the parameter of the system

# Logical state 1

Δ = Δ_opt*γ0
φ = φ_opt*np.pi


θ = np.pi - φ
C = np.array([[-γ0/2, -γ0*np.exp(1.j*φ)/2 , - 1.j*J ],[-γ0*np.exp(1.j*φ)/2,-γ0/2, - 1.j*J ],[- 1.j*J,- 1.j*J, -1.j*Δ ]])
c0 = np.array([0.5, 0.5*np.exp(1.j*θ), - np.sqrt(0.5)])
C1, C2, C3 = C_evolution(C,c0,times)
α_L = np.abs( np.sqrt(γ0/2) *( C1+np.exp( 1.j*φ)*C2) )**2
α_R = np.abs( np.sqrt(γ0/2) *(C1+np.exp(-1.j*φ)*C2) )**2

α_LR = ( np.sqrt(γ0/2) *( C1+np.exp( 1.j*φ)*C2) )*( np.sqrt(γ0/2) *(C1+np.exp(-1.j*φ)*C2) )

I_l = np.trapezoid(α_L, times)
I_r = np.trapezoid(α_R, times)
αL = [α_L]
αR = [α_R]
print([I_r-I_l/(I_l+I_r),(I_l+I_r)])


c0 = np.array([0.5, 0.5*np.exp(1.j*θ), np.sqrt(0.5)])
C1, C2, C3 = C_evolution(C,c0,times)
α_L = np.abs( np.sqrt(γ0/2) *( C1+np.exp( 1.j*φ)*C2) )**2
α_R = np.abs( np.sqrt(γ0/2) *(C1+np.exp(-1.j*φ)*C2) )**2

I_l = np.trapezoid(α_L, times)
I_r = np.trapezoid(α_R, times)
αL.append(α_L)
αR.append(α_R)
print([I_r-I_l/(I_l+I_r),(I_l+I_r)])

In [ ]:

figsize = (8.5, 6)
fig, axs = plt.subplots(2, 2, figsize=figsize)
fig.subplots_adjust(hspace=0.5, wspace=0.4)
csfont = {"fontname": "Times New Roman"}
Legendfont = font_manager.FontProperties(math_fontfamily="cm", size=16)
Labels_FontSize = 18
Box_width = 2

Colors= [
        ["#a8000e", "#ee708b"],
        ["#08589e", "#4eb3d3"],
        ["#006600", "#99FF99"],
        ["#cc4c02", "#fe9929"],
        ["black", "#E0E0E0"],
        ["#990000", "#FFCCCC"]
        ]
Marks = ["o", "s", "D", "p", "8", "v", "p", "v", "^", "8"]
M_size = [6,6,6,6]


l1, l2 = 0,0
#axs[l1][l2].plot(times*γ0, np.exp(-times*γ0/2), linewidth=3, ls="--",color='gray')
ind = 0
axs[l1][l2].plot(times*γ0, np.sqrt(αL[0])/γ0, label=r"$\alpha_{L}$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=600)
ind = 1
axs[l1][l2].plot(times*γ0, np.sqrt(αR[0])/γ0, label=r"$\alpha_{R}$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=900)

l1, l2 = 0,1
#axs[l1][l2].plot(times*γ0, np.exp(-times*γ0/2), linewidth=3, ls="--",color='gray')
ind = 0
axs[l1][l2].plot(times*γ0, np.sqrt(αL[1])/γ0, label=r"$|\eta|\approx 0.995$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=900)
ind = 1
axs[l1][l2].plot(times*γ0, np.sqrt(αR[1])/γ0, label=r"$|\alpha_{R}|$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=600)

l1, l2 = 1,1
#axs[l1][l2].plot(times*γ0, np.exp(-times*γ0/2), linewidth=3, ls="--",color='gray')
ind = 0
axs[l1][l2].plot(times*γ0, α_LR/γ0, label=r"$|\eta|\approx 0.995$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=90)


for l1 in [0, 1]:
    for l2 in [0, 1]:
        for label in axs[l1][l2].get_xticklabels() + axs[l1][l2].get_yticklabels():
            label.set_fontname("Times New Roman")
        axs[l1][l2].legend(prop=Legendfont, loc="best")
        axs[l1][l2].set_xlabel(r"$t \gamma_{0}$", fontsize=Labels_FontSize, math_fontfamily="cm", **csfont)
        axs[l1][l2].set_ylabel(r"$|\alpha|/\sqrt{\gamma_{0}}$", fontsize=Labels_FontSize, math_fontfamily="cm", **csfont)
        axs[l1][l2].xaxis.set_tick_params(width=Box_width, length=6, labelsize=Labels_FontSize)
        axs[l1][l2].yaxis.set_tick_params(width=Box_width, length=6, labelsize=Labels_FontSize)
        axs[l1][l2].tick_params(axis="y", which="minor", width=Box_width, length=4)
        axs[l1][l2].tick_params(axis="x", which="minor", width=Box_width, length=4)
        axs[l1][l2].set_ylim(1e-3, 1e0)
        axs[l1][l2].set_xlim(-8, 257)
        #axs[l1][l2].yaxis.set_major_locator(MultipleLocator(0.1))
        axs[l1][l2].xaxis.set_major_locator(MultipleLocator(50))
        axs[l1][l2].xaxis.set_major_formatter("{x:.0f}")
        axs[l1][l2].set_yscale('log')
        axs[l1][l2].yaxis.set_major_locator(LogLocator(base = 10, numticks = 15))
        #axs[l1][l2].set_xscale('log')
        #axs[l1][l2].yaxis.set_major_formatter("{x:.3f}")
        plt.setp(axs[l1][l2].spines.values(), linewidth = Box_width)

plt.savefig("Correlation.pdf")

## 2.2. The perfect chiral emission

In [ ]:
# objective function
def fidelity_objective(params, L0, HL, psi_L):
    alpha, th1, th2, th3 = params
    
    # evolution
    c0 = L0.copy()
    c0 = expm(-1j * alpha * HL) @ c0
    
    phases_local = np.array([
        np.exp(1j * th1),
        np.exp(1j * th2),
        np.exp(1j * th3)
    ])
    
    c0 = phases_local * c0
    
    # overlap
    overlap = np.vdot(c0, psi_L)
    F = np.abs(overlap)**2
    
    return -F

In [ ]:
#State psi_L

x0 = np.zeros(4)  # initial guess: alpha, theta1, theta2, theta3

L1, L0 = np.array([1/2, 1/2, 1/np.sqrt(2)]), np.array([1/2, 1/2, -1/np.sqrt(2)])

I = 1.j

HL = np.outer(L0, L1) + np.outer(L1, L0) + np.outer(L0, L0) - np.outer(L1, L1)

# This optimal state is obtained from the circuit approach considered in the manuscript.
psi_R = np.array([0.504878 + 0. * I, -0.48546 + 0.138675 *I, -0.693375 - 0.097092 *I])

result_0 = minimize(
    fidelity_objective,
    x0,
    args=(L0, HL, psi_R),
    method='Nelder-Mead'
)

α_opt0, θ1_opt0, θ2_opt0, θ3_opt0 = result_0.x
F_max0 = -result_0.fun
print(F_max0)

# This optimal state is obtained from the circuit approach considered in the manuscript.
psi_L = np.array([0.504878 + 0.* I, - 0.48546 - 0.138675* I, 0.640039 + 0.283807* I])

result_1 = minimize(
    fidelity_objective,
    x0,
    args=(L1, HL, psi_L),
    method='Nelder-Mead'
)

α_opt1, θ1_opt1, θ2_opt1, θ3_opt1 = result_1.x
F_max1 = -result_1.fun
print(F_max1)

In [ ]:
#Defining the parameter of the system

# Logical state 1
γ0 = 0.1
J = 0.1*γ0
times = np.linspace(0,500,30000)/γ0

Δ = -0.131842*γ0
φ = 0.0885687*np.pi


θ = np.pi - φ
C = np.array([[-γ0/2, -γ0*np.exp(1.j*φ)/2 , - 1.j*J ],[-γ0*np.exp(1.j*φ)/2,-γ0/2, - 1.j*J ],[- 1.j*J,- 1.j*J, -1.j*Δ ]])
c0 = L0

# Operations
α,θ1,θ2,θ3 = α_opt0, θ1_opt0, θ2_opt0, θ3_opt0


c0 = expm(-I*α*HL) @ c0
phases_local = np.array([np.exp(I*θ1),np.exp(I*θ2),np.exp(I*θ3)])
c0 = phases_local* c0

C1, C2, C3 = C_evolution(C,c0,times)
α_L = np.abs( np.sqrt(γ0/2) *( C1+np.exp( 1.j*φ)*C2) )**2
α_R = np.abs( np.sqrt(γ0/2) *(C1+np.exp(-1.j*φ)*C2) )**2

α_LR = ( np.sqrt(γ0/2) *( C1+np.exp( 1.j*φ)*C2) )*( np.sqrt(γ0/2) *(C1+np.exp(-1.j*φ)*C2) )

I_l = np.trapezoid(α_L, times)
I_r = np.trapezoid(α_R, times)
αL = [α_L]
αR = [α_R]
print([I_r-I_l/(I_l+I_r),(I_l+I_r)])


α,θ1,θ2,θ3 = α_opt1, θ1_opt1, θ2_opt1, θ3_opt1
c0 = L1
c0 = expm(-I*α*HL) @ c0

phases_local = np.array([np.exp(I*θ1),np.exp(I*θ2),np.exp(I*θ3)])
c0 = phases_local* c0
C1, C2, C3 = C_evolution(C,c0,times)
α_L = np.abs( np.sqrt(γ0/2) *( C1+np.exp( 1.j*φ)*C2) )**2
α_R = np.abs( np.sqrt(γ0/2) *(C1+np.exp(-1.j*φ)*C2) )**2

I_l = np.trapezoid(α_L, times)
I_r = np.trapezoid(α_R, times)
αL.append(α_L)
αR.append(α_R)
print([I_r-I_l/(I_l+I_r),(I_l+I_r)])

eta = I_r-I_l/(I_l+I_r)



In [ ]:
print(np.round(np.array([α_opt0, θ1_opt0, θ2_opt0, θ3_opt0])/np.pi,3))
print(np.round(np.array([α_opt1, θ1_opt1, θ2_opt1, θ3_opt1])/np.pi,3))

In [ ]:
figsize = (8.5, 6)
fig, axs = plt.subplots(2, 2, figsize=figsize)
fig.subplots_adjust(hspace=0.5, wspace=0.4)
csfont = {"fontname": "Times New Roman"}
Legendfont = font_manager.FontProperties(math_fontfamily="cm", size=16)
Labels_FontSize = 18
Box_width = 2

Colors= [
        ["#a8000e", "#ee708b"],
        ["#08589e", "#4eb3d3"],
        ["#006600", "#99FF99"],
        ["#cc4c02", "#fe9929"],
        ["black", "#E0E0E0"],
        ["#990000", "#FFCCCC"]
        ]
Marks = ["o", "s", "D", "p", "8", "v", "p", "v", "^", "8"]
M_size = [6,6,6,6]


l1, l2 = 0,0
#axs[l1][l2].plot(times*γ0, np.exp(-times*γ0/2), linewidth=3, ls="--",color='gray')
ind = 0
axs[l1][l2].plot(times*γ0, np.sqrt(αL[0])/γ0, label=r"$|\alpha_{L}|$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=600)
ind = 1
axs[l1][l2].plot(times*γ0, np.sqrt(αR[0])/γ0, label=r"$|\alpha_{R}|$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=900)

l1, l2 = 0,1
#axs[l1][l2].plot(times*γ0, np.exp(-times*γ0/2), linewidth=3, ls="--",color='gray')
ind = 0
axs[l1][l2].plot(times*γ0, np.sqrt(αL[1])/γ0, label=r"$|\alpha_{L}|$" %abs(eta), linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=900)
ind = 1
axs[l1][l2].plot(times*γ0, np.sqrt(αR[1])/γ0, label=r"$|\alpha_{R}|$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=600)

l1, l2 = 1,1
#axs[l1][l2].plot(times*γ0, np.exp(-times*γ0/2), linewidth=3, ls="--",color='gray')
ind = 0
axs[l1][l2].plot(times*γ0, α_LR/γ0, label=r"$|\eta|\approx 0.995$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=90)


for l1 in [0, 1]:
    for l2 in [0, 1]:
        for label in axs[l1][l2].get_xticklabels() + axs[l1][l2].get_yticklabels():
            label.set_fontname("Times New Roman")
        axs[l1][l2].legend(prop=Legendfont, loc="best")
        axs[l1][l2].set_xlabel(r"$t \gamma_{0}$", fontsize=Labels_FontSize, math_fontfamily="cm", **csfont)
        axs[l1][l2].set_ylabel(r"$|\alpha|/\sqrt{\gamma_{0}}$", fontsize=Labels_FontSize, math_fontfamily="cm", **csfont)
        axs[l1][l2].xaxis.set_tick_params(width=Box_width, length=6, labelsize=Labels_FontSize)
        axs[l1][l2].yaxis.set_tick_params(width=Box_width, length=6, labelsize=Labels_FontSize)
        axs[l1][l2].tick_params(axis="y", which="minor", width=Box_width, length=4)
        axs[l1][l2].tick_params(axis="x", which="minor", width=Box_width, length=4)
        axs[l1][l2].set_ylim(-0.02, np.sqrt(0.5))
        axs[l1][l2].set_xlim(-8, 257)
        axs[l1][l2].yaxis.set_major_locator(MultipleLocator(0.2))
        axs[l1][l2].xaxis.set_major_locator(MultipleLocator(50))
        axs[l1][l2].xaxis.set_major_formatter("{x:.0f}")
        #axs[l1][l2].set_yscale('log')
        #axs[l1][l2].yaxis.set_major_locator(LogLocator(base = 10, numticks = 15))
        #axs[l1][l2].set_xscale('log')
        #axs[l1][l2].yaxis.set_major_formatter("{x:.3f}")
        plt.setp(axs[l1][l2].spines.values(), linewidth = Box_width)

plt.savefig("FullDirectionality.pdf")

### 2.5.a 2D map optimization

In [ ]:
#Defining the parameter of the system

print("Warning: This is a grid plot, which may take time to be finished")


γ0 = 0.1
J = 0.1*γ0
Δ_opt = -0.131842
φ_opt = 0.0885687
times = np.linspace(0,500,30000)/γ0
Δ_list = np.linspace(-0.4 , 0.0,81) *γ0#
Δ_list = np.sort(np.append(Δ_list, Δ_opt*γ0))
φ_list = np.linspace( 0,0.25,51)*np.pi #
φ_list = np.sort(np.append(φ_list, φ_opt*np.pi))

η_R_perf = []
αL_perf, αR_perf = [], []
for Δ in Δ_list:
    α_LΔ, α_RΔ, η_Δ = [], [], []
    for φ in φ_list:
        C = np.array([[-γ0/2, -γ0*np.exp(1.j*φ)/2 , - 1.j*J ],[-γ0*np.exp(1.j*φ)/2,-γ0/2, - 1.j*J ],[- 1.j*J,- 1.j*J, -1.j*Δ ]])
        c0 = L0
        # Operations
        α,θ1,θ2,θ3 = α_opt0, θ1_opt0, θ2_opt0, θ3_opt0

        c0 = expm(-I*α*HL) @ c0
        phases_local = np.array([np.exp(I*θ1),np.exp(I*θ2),np.exp(I*θ3)])
        c0 = phases_local* c0
        C1, C2, C3 = C_evolution(C,c0,times)
        α_L = np.abs( np.sqrt(γ0/2) *( C1+np.exp( 1.j*φ)*C2) )**2
        α_R = np.abs( np.sqrt(γ0/2) *(C1+np.exp(-1.j*φ)*C2) )**2

        I_l = np.trapezoid(α_L, times)
        I_r = np.trapezoid(α_R, times)
        α_LΔ.append(α_L)
        α_RΔ.append(α_R)
        η_Δ.append((I_r-I_l)/(I_l+I_r))
    η_R_perf.append(η_Δ)
    αL_perf.append(α_LΔ)
    αR_perf.append(α_RΔ)


In [ ]:
η_R_perf = np.array(η_R_perf)

Δ_listind, φ_optind = np.where(η_R_perf == np.max(η_R_perf))
η_R_perfmax = η_R_perf[Δ_listind[0]][φ_optind[0]]
print(η_R_perfmax)
print(Δ_listind, φ_optind)

In [ ]:

figsize = (8, 6)
fig, axs = plt.subplots(2, 2, figsize=figsize)
fig.subplots_adjust(hspace=0.5, wspace=0.4)
csfont = {"fontname": "Times New Roman"}
Legendfont = font_manager.FontProperties(math_fontfamily="cm", size=16)
Labels_FontSize = 18
Box_width = 2

Colors= [
        ["#08589e", "#4eb3d3"],
        ["#91003f", "#c994c7"],
        ["#006600", "#99FF99"],
        ["#cc4c02", "#fe9929"],
        ["black", "#E0E0E0"],
        ["#990000", "#FFCCCC"]
        ]
Marks = ["o", "s", "D", "p", "8", "v", "p", "v", "^", "8"]
M_size = [6,6,6,6]


l1, l2 = 0,0

imR = axs[l1][l2].pcolormesh(φ_list/np.pi, Δ_list/γ0 ,η_R_perf, cmap='coolwarm_r', vmin=-1.0, vmax=1.0,rasterized=True)
axs[l1][l2].axvline(φ_list[φ_optind]/np.pi,color = 'black', ls = '--')
axs[l1][l2].axhline(Δ_list[Δ_listind]/γ0,color = 'black', ls = '--')
axs[l1][l2].text( 0.5, 0.95, r'$\eta_{\mathrm{opt}} = %.2f$' %(η_R_perfmax), transform=axs[l1][l2].transAxes,fontsize=17,va='top', ha='left', color='black', math_fontfamily="cm", **csfont)


for l1 in [0, 1]:
    for l2 in [0, 1]:
        for label in axs[l1][l2].get_xticklabels() + axs[l1][l2].get_yticklabels():
            label.set_fontname("Times New Roman")
        axs[l1][l2].set_xlabel(r"$\phi/\pi$", fontsize=Labels_FontSize, math_fontfamily="cm", **csfont)
        axs[l1][l2].set_ylabel(r"$\Delta/\gamma_{0}$", fontsize=Labels_FontSize, math_fontfamily="cm", **csfont)
        axs[l1][l2].xaxis.set_tick_params(width=Box_width, length=6, labelsize=Labels_FontSize)
        axs[l1][l2].yaxis.set_tick_params(width=Box_width, length=6, labelsize=Labels_FontSize)
        axs[l1][l2].tick_params(axis="y", which="minor", width=Box_width, length=4)
        axs[l1][l2].tick_params(axis="x", which="minor", width=Box_width, length=4)
        axs[l1][l2].set_ylim(-0.4, 0.0)
        axs[l1][l2].set_xlim(0.0, 0.2)
        axs[l1][l2].yaxis.set_major_locator(MultipleLocator(0.1))
        axs[l1][l2].yaxis.set_major_formatter("{x:.1f}")
        axs[l1][l2].xaxis.set_major_locator(MultipleLocator(0.05))
        axs[l1][l2].xaxis.set_major_formatter("{x:.2f}")
        
        #axs[l1][l2].set_xscale('log')
        #axs[l1][l2].yaxis.set_major_formatter("{x:.3f}")
        plt.setp(axs[l1][l2].spines.values(), linewidth = Box_width)


cbar11 = fig.colorbar(imR, ax=axs[1][1], fraction=0.046, pad=0.04)
cbar11.set_label(r'$\mathrm{Directionality~Param.},~\eta$', fontsize=Labels_FontSize, math_fontfamily="cm", **csfont)
cbar11.set_ticks([-1.0, -0.5, 0.0, 0.5, 1.0])  # choose positions
cbar11.ax.tick_params(labelsize=Labels_FontSize, width=Box_width, length=4)
for label in cbar11.ax.get_yticklabels():
    label.set_fontname("Times New Roman")
plt.savefig("Directional_Perfect2.pdf")

## 2.3. Imperfect measurements

In [ ]:
#Here we use the same parameters as before, but not we consider the comparison with the logical 1

print("Warning: This is a grid plot, which may take time to be finished")

Δ_optP = -0.131842*γ0
φ_opt = 0.0885687
J = 0.1*γ0

δΔ_listP = np.linspace(-0.1 , 0.1,51) *Δ_optP
δθ_list = np.linspace(-0.1 , 0.1,51)*np.pi 
L0 = np.array([0.5, 0.5, -np.sqrt(0.5)])
L1 = np.array([0.5, 0.5, np.sqrt(0.5)])
HL = np.outer(L0, L1) + np.outer(L1, L0) + np.outer(L0, L0) - np.outer(L1, L1)
α,θ1,θ2,θ3 = α_opt0, θ1_opt0, θ2_opt0, θ3_opt0

E_0Perf = []
for δΔ in δΔ_listP:
    I_RΔ = []
    for δθ in δθ_list:
        phases_local = np.array([np.exp(I*(θ1 + δθ)),np.exp(I*(θ2 + δθ)),np.exp(I*(θ3 + δθ))])
        
        Δ = Δ_optP + δΔ
        φ = φ_opt*np.pi
        C = np.array([[-γ0/2, -γ0*np.exp(1.j*φ)/2 , - 1.j*J ],[-γ0*np.exp(1.j*φ)/2,-γ0/2, - 1.j*J ],[- 1.j*J,- 1.j*J, -1.j*Δ ]])
        #E(0,0)
        
        c0 = expm(-I*(α+δθ)*HL) @ L0
        
        c0 = phases_local* c0
        C1, C2, C3 = C_evolution(C,c0,times)
        α_R = np.abs( np.sqrt(γ0/2) *(C1+np.exp(-1.j*φ)*C2) )**2
        I_r = np.trapezoid(α_R, times)
        I_RΔ.append(1-I_r)
    E_0Perf.append(I_RΔ)
E_0Perf = np.array(E_0Perf)

E_1Perf = []
α,θ1,θ2,θ3 = α_opt1, θ1_opt1, θ2_opt1, θ3_opt1
for δΔ in δΔ_listP:
    I_LΔ = [ ]
    for δθ in δθ_list:
        phases_local = np.array([np.exp(I*(θ1 + δθ)),np.exp(I*(θ2 + δθ)),np.exp(I*(θ3 + δθ))])
        Δ = Δ_optP + δΔ
        φ = φ_opt*np.pi
        C = np.array([[-γ0/2, -γ0*np.exp(1.j*φ)/2 , - 1.j*J ],[-γ0*np.exp(1.j*φ)/2,-γ0/2, - 1.j*J ],[- 1.j*J,- 1.j*J, -1.j*Δ ]])
        #E(0,0)
        
        c0 = expm(-I*(α+δθ)*HL) @ L1
        
        c0 = phases_local* c0
        C1, C2, C3 = C_evolution(C,c0,times)
        α_L = np.abs( np.sqrt(γ0/2) *(C1+np.exp(1.j*φ)*C2) )**2
        I_l = np.trapezoid(α_L, times)
        I_LΔ.append(1-I_l)
    E_1Perf.append(I_LΔ)
E_1Perf = np.array(E_1Perf)

In [ ]:
x0 = [0.0, 0.0]  # initial guess: phi, Delta
γ0 = 0.1
J = 0.1*γ0
times = np.linspace(0,300,3000)/γ0

result = minimize(
    objective,
    x0,
    args=(γ0, J, times),
    method='Nelder-Mead'
)

φ_opt, Δ_opt= result.x
eta_max = -result.fun

φ_opt = φ_opt/np.pi
Δ_opt = Δ_opt/γ0
print(eta_max)

print('Optimal φ =', φ_opt)
print('Optimal Δ =', Δ_opt)

## 2.4. State-Dependent Directionality

### The new logical basis 

In [ ]:
#State psi_L

L1, L0 = np.array([1/2, 1/2, 1/np.sqrt(2)]), np.array([1/2, 1/2, -1/np.sqrt(2)])

I = 1.j
psi_L = np.array([0.504878 + 0. * I, -0.48546 + 0.138675 *I, -0.693375 - 0.097092 *I])


In [ ]:
#State psi_L

L1, L0 = np.array([1/2, 1/2, 1/np.sqrt(2)]), np.array([1/2, 1/2, -1/np.sqrt(2)])

I = 1.j
psi_L = np.array([0.504878 + 0. * I, -0.48546 + 0.138675 *I, -0.693375 - 0.097092 *I])

#Transformations for perfect chiral emission

HL = np.outer(L0, L1) + np.outer(L1, L0) + np.outer(L0, L0) - np.outer(L1, L1)

a = 0.031624*np.pi
psi_evolve0 = expm(-I*a*HL) @ L0

print("")
theta1 = 0.1257*np.pi/2
print(psi_R[0]-psi_evolve0[0]*np.exp(I*theta1))

print("")
theta2 = (1+0.151408)*np.pi
print(psi_R[1] - psi_evolve0[1]*np.exp(I*theta2))


print("")
theta3 = (1+0.1328515)*np.pi
print(psi_R[2] - psi_evolve0[2]*np.exp(I*theta3))


print("")
print("")
print("")
phases_local = np.array([np.exp(I*theta1),np.exp(I*theta2),np.exp(I*theta3)])

print(phases_local*psi_evolve0 - psi_R)



In [ ]:
#Defining the parameter of the system

# Logical state 1
γ0 = 0.1
J = 0.1*γ0
times = np.linspace(0,500,30000)/γ0

Δ = -0.131842*γ0
φ = 0.0885687*np.pi


θ = np.pi - φ
C = np.array([[-γ0/2, -γ0*np.exp(1.j*φ)/2 , - 1.j*J ],[-γ0*np.exp(1.j*φ)/2,-γ0/2, - 1.j*J ],[- 1.j*J,- 1.j*J, -1.j*Δ ]])
c0 = L0

# Operations
α = 0.031624*np.pi
θ1 = 0.1251*np.pi/2
θ2 = 1.151408*np.pi
θ3 = 0.1328515*np.pi


c0 = expm(-I*α*HL) @ c0
phases_local = np.array([np.exp(I*θ1),np.exp(I*θ2),np.exp(I*θ3)])
c0 = phases_local* c0

C1, C2, C3 = C_evolution(C,c0,times)
α_L = np.abs( np.sqrt(γ0/2) *( C1+np.exp( 1.j*φ)*C2) )**2
α_R = np.abs( np.sqrt(γ0/2) *(C1+np.exp(-1.j*φ)*C2) )**2

α_LR = ( np.sqrt(γ0/2) *( C1+np.exp( 1.j*φ)*C2) )*( np.sqrt(γ0/2) *(C1+np.exp(-1.j*φ)*C2) )

I_l = np.trapezoid(α_L, times)
I_r = np.trapezoid(α_R, times)
αL = [α_L]
αR = [α_R]
print([I_r-I_l/(I_l+I_r),(I_l+I_r)])


c0 = L1

# Operations
α = 0.031624*np.pi
c0 = expm(-I*α*HL) @ c0

θ1 = 0.1251*np.pi/2
θ2 = 1.151408*np.pi
θ3 = (0.1328515)*np.pi
phases_local = np.array([np.exp(I*θ1),np.exp(I*θ2),np.exp(I*θ3)])
c0 = phases_local* c0
C1, C2, C3 = C_evolution(C,c0,times)
α_L = np.abs( np.sqrt(γ0/2) *( C1+np.exp( 1.j*φ)*C2) )**2
α_R = np.abs( np.sqrt(γ0/2) *(C1+np.exp(-1.j*φ)*C2) )**2

I_l = np.trapezoid(α_L, times)
I_r = np.trapezoid(α_R, times)
αL.append(α_L)
αR.append(α_R)
print([I_r-I_l/(I_l+I_r),(I_l+I_r)])

eta = I_r-I_l/(I_l+I_r)

figsize = (8.5, 6)
fig, axs = plt.subplots(2, 2, figsize=figsize)
fig.subplots_adjust(hspace=0.5, wspace=0.4)
csfont = {"fontname": "Times New Roman"}
Legendfont = font_manager.FontProperties(math_fontfamily="cm", size=16)
Labels_FontSize = 18
Box_width = 2

Colors= [
        ["#a8000e", "#ee708b"],
        ["#08589e", "#4eb3d3"],
        ["#006600", "#99FF99"],
        ["#cc4c02", "#fe9929"],
        ["black", "#E0E0E0"],
        ["#990000", "#FFCCCC"]
        ]
Marks = ["o", "s", "D", "p", "8", "v", "p", "v", "^", "8"]
M_size = [6,6,6,6]


l1, l2 = 0,0
#axs[l1][l2].plot(times*γ0, np.exp(-times*γ0/2), linewidth=3, ls="--",color='gray')
ind = 0
axs[l1][l2].plot(times*γ0, np.sqrt(αL[0])/γ0, label=r"$|\alpha_{L}|$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=600)
ind = 1
axs[l1][l2].plot(times*γ0, np.sqrt(αR[0])/γ0, label=r"$|\alpha_{R}|$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=900)

l1, l2 = 0,1
#axs[l1][l2].plot(times*γ0, np.exp(-times*γ0/2), linewidth=3, ls="--",color='gray')
ind = 0
axs[l1][l2].plot(times*γ0, np.sqrt(αL[1])/γ0, label=r"$|\alpha_{L}|$" %abs(eta), linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=900)
ind = 1
axs[l1][l2].plot(times*γ0, np.sqrt(αR[1])/γ0, label=r"$|\alpha_{R}|$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=600)

l1, l2 = 1,1
#axs[l1][l2].plot(times*γ0, np.exp(-times*γ0/2), linewidth=3, ls="--",color='gray')
ind = 0
axs[l1][l2].plot(times*γ0, α_LR/γ0, label=r"$|\eta|\approx 0.995$", linewidth=3, ls="-", color=Colors[ind][0], 
                     marker=Marks[ind], markersize=M_size[ind], mec=Colors[ind][0], mfc=Colors[ind][1], markeredgewidth=1, markevery=90)


for l1 in [0, 1]:
    for l2 in [0, 1]:
        for label in axs[l1][l2].get_xticklabels() + axs[l1][l2].get_yticklabels():
            label.set_fontname("Times New Roman")
        axs[l1][l2].legend(prop=Legendfont, loc="best")
        axs[l1][l2].set_xlabel(r"$t \gamma_{0}$", fontsize=Labels_FontSize, math_fontfamily="cm", **csfont)
        axs[l1][l2].set_ylabel(r"$|\alpha|/\sqrt{\gamma_{0}}$", fontsize=Labels_FontSize, math_fontfamily="cm", **csfont)
        axs[l1][l2].xaxis.set_tick_params(width=Box_width, length=6, labelsize=Labels_FontSize)
        axs[l1][l2].yaxis.set_tick_params(width=Box_width, length=6, labelsize=Labels_FontSize)
        axs[l1][l2].tick_params(axis="y", which="minor", width=Box_width, length=4)
        axs[l1][l2].tick_params(axis="x", which="minor", width=Box_width, length=4)
        axs[l1][l2].set_ylim(-0.02, np.sqrt(0.5))
        axs[l1][l2].set_xlim(-8, 257)
        #axs[l1][l2].yaxis.set_major_locator(MultipleLocator(0.1))
        axs[l1][l2].xaxis.set_major_locator(MultipleLocator(50))
        axs[l1][l2].xaxis.set_major_formatter("{x:.0f}")
        #axs[l1][l2].set_yscale('log')
        #axs[l1][l2].yaxis.set_major_locator(LogLocator(base = 10, numticks = 15))
        #axs[l1][l2].set_xscale('log')
        #axs[l1][l2].yaxis.set_major_formatter("{x:.3f}")
        plt.setp(axs[l1][l2].spines.values(), linewidth = Box_width)

plt.savefig("FullDirectionality_2.pdf")